# Data Cleaning Pipeline

This notebook combines raw data from multiple sources to create the final training dataset:
- Housing prices (wide format)
- Personal income (wide format)
- Mortgage rates (time series)
- Affordability metrics (long format)

In [31]:
import pandas as pd
from pathlib import Path

In [32]:
# path setup 
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [33]:
# load raw data 
affordability_df = pd.read_csv(RAW_DIR / "affordability_county.csv")
housing_prices_df = pd.read_csv(RAW_DIR / "avg_housing_prices_county.csv")
mortgage_rates_df = pd.read_csv(RAW_DIR / "Mortgage_Rates.csv")
income_df = pd.read_csv(RAW_DIR / "personal_income_county.csv")

In [34]:
# transform housing prices df from wide to long

# create county FIPS code
housing_prices_df['fips'] = (
    housing_prices_df['StateCodeFIPS'].astype(int) * 1000 
    + housing_prices_df['MunicipalCodeFIPS'].astype(int)
)

# Date columns
price_columns = [col for col in housing_prices_df.columns if col.count('-') == 2]

# wide to long
housing_long = housing_prices_df[["fips"] + price_columns].melt(
    id_vars="fips",
    var_name="date",
    value_name="avg_housing_price",
)

# standardize to first day of month
housing_long['date'] = pd.to_datetime(housing_long['date']).dt.to_period("M").dt.to_timestamp()

housing_long = housing_long.sort_values(['fips', 'date']).reset_index(drop=True)

housing_long.head()

C:\Users\ryanp\AppData\Local\Temp\ipykernel_26316\4251393781.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  housing_prices_df['fips'] = (


,fips,date,avg_housing_price
0,1001,2000-01-01,122058.081559
1,1001,2000-02-01,122088.584666
2,1001,2000-03-01,121888.329099
3,1001,2000-04-01,121812.354810
4,1001,2000-05-01,121810.006714


In [35]:
# transform income df from wide to long
income_df = income_df.rename(columns={'GeoFIPS': 'fips'})

year_columns = [col for col in income_df.columns if col.isdigit()]

# wide to long
income_long = income_df[['fips'] + year_columns].melt(
    id_vars="fips",
    var_name="year",
    value_name="income"
)

income_long['year'] = income_long['year'].astype(int)
income_long['fips'] = income_long['fips'].astype(int)

income_long = income_long.sort_values(['fips', 'year']).reset_index(drop=True)

income_long.head()

,fips,year,income
0,1001,2000,23584
1,1001,2001,24643
2,1001,2002,25082
3,1001,2003,26500
4,1001,2004,27745


In [36]:
# averge monthly mortgage rates from weekly observations
mortgage_rates_df['observation_date'] = pd.to_datetime(mortgage_rates_df['observation_date'])

mortgage_rates_df['year_month'] = mortgage_rates_df['observation_date'].dt.to_period('M')

mortgage_monthly = mortgage_rates_df.groupby('year_month')['MORTGAGE30US'].mean().reset_index()

mortgage_monthly.columns = ['year_month', 'mortgage_rate']
mortgage_monthly['date'] = mortgage_monthly['year_month'].dt.to_timestamp()
mortgage_monthly = mortgage_monthly[['date', 'mortgage_rate']]

mortgage_monthly.head()

,date,mortgage_rate
0,1971-04-01,7.3100
1,1971-05-01,7.4250
2,1971-06-01,7.5300
3,1971-07-01,7.6040
4,1971-08-01,7.6975


In [37]:
# prepare affordability df

# convert to same timestamp format as housing and mortgage
affordability_df['date'] = (
    pd.to_datetime(affordability_df['Month'])
    .dt.to_period("M")
    .dt.to_timestamp()
)

affordability_df = affordability_df.rename(columns={'County FIP': 'fips'})

afford_for_merge = affordability_df[['fips', 'date', 'Affordability Metric', 'county_perc_mo']].copy()
afford_for_merge["fips"] = afford_for_merge["fips"].astype(int)
afford_for_merge = afford_for_merge.sort_values(['fips', 'date']).reset_index(drop=True)

afford_for_merge.head()

,fips,date,Affordability Metric,county_perc_mo
0,1001,2006-12-01,87.701976,0.342102
1,1001,2007-01-01,91.503678,0.327888
2,1001,2007-02-01,95.883109,0.312912
3,1001,2007-03-01,97.340268,0.308228
4,1001,2007-04-01,108.446069,0.276663


In [38]:
# combine into single df
final_df = afford_for_merge.merge(mortgage_monthly, on='date', how='inner')
final_df = final_df.merge(housing_long, on=['fips', 'date'], how='left')

final_df['year'] = final_df['date'].dt.year
final_df['month'] = final_df['date'].dt.month

final_df = final_df.merge(income_long, on=['fips', 'year'], how='inner')

# order cols
final_df = final_df[
    [
        "fips",
        "Affordability Metric",
        "county_perc_mo",
        "mortgage_rate",
        "avg_housing_price",
        "year",
        "income",
        "month",
    ]
].copy()

final_df = final_df.sort_values(["fips", "year", "month"]).reset_index(drop=True)

final_df.head()

,fips,Affordability Metric,county_perc_mo,mortgage_rate,avg_housing_price,year,income,month
0,1001,87.701976,0.342102,6.1350,163940.199861,2006,29942,12
1,1001,91.503678,0.327888,6.2175,164297.125318,2007,31665,1
2,1001,95.883109,0.312912,6.2850,164967.775699,2007,31665,2
3,1001,97.340268,0.308228,6.1560,165492.145947,2007,31665,3
4,1001,108.446069,0.276663,6.1800,165851.129466,2007,31665,4


In [39]:
# save cleaned dataset
final_df.to_csv(PROCESSED_DIR / "final_3d_df_REVISED.csv", index=False)

print(final_df.shape)

(371244, 8)


In [40]:
# add lag and rolling features
feature_df = final_df.copy()
feature_df = feature_df.sort_values(["fips", "year", "month"]).reset_index(drop=True)

feature_columns = [
    "Affordability Metric",
    "county_perc_mo",
    "mortgage_rate",
    "income",
    "avg_housing_price",
]

grouped = feature_df.groupby("fips", sort=False)

# lag 
for col in feature_columns:
    feature_df[f"{col}_prev_mo"] = grouped[col].shift(1)

# rolling
for col in feature_columns:
    shifted = grouped[col].shift(1)

    feature_df[f"{col}_rolling_3m"] = (
        shifted.groupby(feature_df["fips"])
        .rolling(window=3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    feature_df[f"{col}_rolling_12m"] = (
        shifted.groupby(feature_df["fips"])
        .rolling(window=12, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

generated_features = [
    col for col in feature_df.columns
    if col.endswith("_prev_mo")
    or col.endswith("_rolling_3m")
    or col.endswith("_rolling_12m")
]

feature_df = feature_df.dropna(subset=generated_features).reset_index(drop=True)

feature_df.head()

,fips,Affordability Metric,county_perc_mo,mortgage_rate,avg_housing_price,year,income,month,Affordability Metric_prev_mo,county_perc_mo_prev_mo,...,Affordability Metric_rolling_3m,Affordability Metric_rolling_12m,county_perc_mo_rolling_3m,county_perc_mo_rolling_12m,mortgage_rate_rolling_3m,mortgage_rate_rolling_12m,income_rolling_3m,income_rolling_12m,avg_housing_price_rolling_3m,avg_housing_price_rolling_12m
0,1001,91.503678,0.327888,6.2175,164297.125318,2007,31665,1,87.701976,0.342102,...,87.701976,87.701976,0.342102,0.342102,6.13500,6.135000,29942.000000,29942.000000,163940.199861,163940.199861
1,1001,95.883109,0.312912,6.2850,164967.775699,2007,31665,2,91.503678,0.327888,...,89.602827,89.602827,0.334995,0.334995,6.17625,6.176250,30803.500000,30803.500000,164118.662589,164118.662589
2,1001,97.340268,0.308228,6.1560,165492.145947,2007,31665,3,95.883109,0.312912,...,91.696254,91.696254,0.327634,0.327634,6.21250,6.212500,31090.666667,31090.666667,164401.700293,164401.700293
3,1001,108.446069,0.276663,6.1800,165851.129466,2007,31665,4,97.340268,0.308228,...,94.909018,93.107258,0.316343,0.322783,6.21950,6.198375,31665.000000,31234.250000,164919.015655,164674.311706
4,1001,108.297781,0.277042,6.2620,165675.502071,2007,31665,5,108.446069,0.276663,...,100.556482,96.175020,0.299268,0.313559,6.20700,6.194700,31665.000000,31320.400000,165437.017037,164909.675258


In [41]:
# save
feature_df.to_csv(PROCESSED_DIR / "final_3d_df_with_rolling_averages_REVISED.csv", index=False)